In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc
import os
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from ddw.utils.mrctools import load_mrc_data

C:\Users\chris\anaconda3\envs\ddw_env\lib\site-packages\lightning_utilities\core\imports.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
def load_volume(path):
    vol = load_mrc_data(path)
    if isinstance(vol, torch.Tensor):
        vol = vol.cpu().numpy()
    return vol.astype(np.float32)

### Loading Data Inference exp

In [9]:
results_dir = Path("../../results/angle_sweep")
angles = [30, 50, 70]

refined_paths = {
    angle: results_dir / f"tomo_even_frames+tomo_odd_frames_mw{angle}_avg_refined.rec"
    for angle in angles
}

refined_avg = {}

for angle, path in refined_paths.items():
    refined_avg[angle] = load_volume(path)

### Loading Data Training exp

In [10]:
results_dir_02 = Path("../../experiments/subexp02")
angles_train = [40, 50, 60]

refined_paths_02 = {
    angle: results_dir_02 / f"refine_mw{angle}/tomo_even_frames+tomo_odd_frames_train{angle}_mw50_avg_refined.rec"
    for angle in angles_train
}

for angle, path in refined_paths_02.items():
    print(angle, path.exists(), path)
    

refined_avg_02 = {}

for angle, path in refined_paths_02.items():
    refined_avg_02[angle] = load_volume(path)

40 True ..\..\experiments\subexp02\refine_mw40\tomo_even_frames+tomo_odd_frames_train40_mw50_avg_refined.rec
50 True ..\..\experiments\subexp02\refine_mw50\tomo_even_frames+tomo_odd_frames_train50_mw50_avg_refined.rec
60 True ..\..\experiments\subexp02\refine_mw60\tomo_even_frames+tomo_odd_frames_train60_mw50_avg_refined.rec


In [11]:
metrics_out_dir = Path("../../figures/final_report/metrics")
metrics_out_dir.mkdir(parents=True, exist_ok=True)

cases = [
    {
        "experiment": "subexp01",
        "case": "30_vs_50",
        "description": "refine30 vs refine50 baseline",
        "case_vol": refined_avg[30],
        "baseline_vol": refined_avg[50],
    },
    {
        "experiment": "subexp01",
        "case": "70_vs_50",
        "description": "refine70 vs refine50 baseline",
        "case_vol": refined_avg[70],
        "baseline_vol": refined_avg[50],
    },
    {
        "experiment": "subexp02",
        "case": "train40_vs_train50",
        "description": "train40/refine50 vs train50/refine50 baseline",
        "case_vol": refined_avg_02[40],
        "baseline_vol": refined_avg_02[50],
    },
    {
        "experiment": "subexp02",
        "case": "train60_vs_train50",
        "description": "train60/refine50 vs train50/refine50 baseline",
        "case_vol": refined_avg_02[60],
        "baseline_vol": refined_avg_02[50],
    },
]

In [12]:
def real_space_metrics(case_vol, baseline_vol):
    """
    Compute whole-volume absolute real-space difference metrics.
    """
    diff = np.abs(case_vol.astype(np.float32) - baseline_vol.astype(np.float32))

    out = {
        "real_mean_abs_diff": float(np.mean(diff)),
        "real_median_abs_diff": float(np.median(diff)),
        "real_p95_abs_diff": float(np.percentile(diff, 95)),
        "real_p99_abs_diff": float(np.percentile(diff, 99)),
        "real_max_abs_diff": float(np.max(diff)),
    }

    del diff
    gc.collect()

    return out


def fourier_logpower_metrics(case_vol, baseline_vol):
    """
    Compute absolute log-power difference metrics in Fourier space.

    This uses the full 3D volume, but processes one case at a time.
    """
    case_vol = case_vol.astype(np.float32, copy=False)
    baseline_vol = baseline_vol.astype(np.float32, copy=False)

    F_case = np.fft.fftn(case_vol)
    F_base = np.fft.fftn(baseline_vol)

    logpower_case = np.log1p(np.abs(F_case) ** 2)
    logpower_base = np.log1p(np.abs(F_base) ** 2)

    diff = np.abs(logpower_case - logpower_base)

    out = {
        "fourier_mean_abs_logpower_diff": float(np.mean(diff)),
        "fourier_median_abs_logpower_diff": float(np.median(diff)),
        "fourier_p95_abs_logpower_diff": float(np.percentile(diff, 95)),
        "fourier_p99_abs_logpower_diff": float(np.percentile(diff, 99)),
        "fourier_max_abs_logpower_diff": float(np.max(diff)),
    }

    del F_case, F_base, logpower_case, logpower_base, diff
    gc.collect()

    return out

In [13]:
rows = []

for c in cases:
    print("Computing:", c["experiment"], c["case"])

    row = {
        "experiment": c["experiment"],
        "case": c["case"],
        "description": c["description"],
    }

    row.update(real_space_metrics(c["case_vol"], c["baseline_vol"]))
    row.update(fourier_logpower_metrics(c["case_vol"], c["baseline_vol"]))

    rows.append(row)

metrics_df = pd.DataFrame(rows)

save_csv = metrics_out_dir / "Difference_Metrics_Summary.csv"
metrics_df.to_csv(save_csv, index=False)

display(metrics_df)

print("Saved:", save_csv)

Computing: subexp01 30_vs_50
Computing: subexp01 70_vs_50
Computing: subexp02 train40_vs_train50
Computing: subexp02 train60_vs_train50


,experiment,case,description,real_mean_abs_diff,real_median_abs_diff,real_p95_abs_diff,real_p99_abs_diff,real_max_abs_diff,fourier_mean_abs_logpower_diff,fourier_median_abs_logpower_diff,fourier_p95_abs_logpower_diff,fourier_p99_abs_logpower_diff,fourier_max_abs_logpower_diff
0,subexp01,30_vs_50,refine30 vs refine50 baseline,0.069655,0.057953,0.173717,0.233901,2.471838,0.816988,0.567099,2.464372,4.017154,14.423241
1,subexp01,70_vs_50,refine70 vs refine50 baseline,0.091519,0.075269,0.229866,0.317780,7.810798,1.229872,0.930538,3.398211,5.009711,15.213666
2,subexp02,train40_vs_train50,train40/refine50 vs train50/refine50 baseline,0.065948,0.051189,0.175952,0.267337,6.840391,1.028100,0.729681,3.022749,4.624955,15.568051
3,subexp02,train60_vs_train50,train60/refine50 vs train50/refine50 baseline,0.082788,0.065928,0.213725,0.313882,10.875136,1.131361,0.838407,3.200873,4.810244,15.258960


Saved: ..\..\figures\final_report\metrics\Difference_Metrics_Summary.csv
